# 04 — Data transformation and aggregation

## Objective
Reshape cleaned point-level observations and derive the available standardized city-level product. Because the input is monthly, the aggregation grain is city-month rather than city-day. The output includes the standardized `city` field; `state` is a state/territory label, and district remains unavailable.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name.lower()=="notebooks" else Path.cwd()
sys.path.insert(0,str(ROOT/"src"))
from climate_pipeline.config import RAW_DIR, PROJECT_DIR, FINAL_DAILY_COLUMNS
from climate_pipeline import run_pipeline, validate_city_monthly, validate_daily_schema
pd.set_option("display.max_columns",60)


## Mapping and point-level transformation

The source-to-target mapping is documented with units and interpretation. `PRECTOTCORR` is a monthly mean daily precipitation rate in mm/day, not monthly accumulated rainfall. `T2M`, `T2M_MAX`, and `T2M_MIN` are monthly summaries. All 15 source parameter columns are preserved in the processed table.

In [2]:
from climate_pipeline.ingest import discover_files,read_all_points
from climate_pipeline.transform import point_monthly_wide,build_city_monthly,build_data_dictionary
inventory=discover_files(RAW_DIR)
point_long,annual=read_all_points(inventory,RAW_DIR)
point_wide=point_monthly_wide(point_long)
city_monthly=build_city_monthly(point_wide,inventory)
dictionary=build_data_dictionary(point_long,city_monthly)
print("Point long",point_long.shape,"Point wide",point_wide.shape,"City-month",city_monthly.shape)
display(dictionary)


Point long (416520, 11) Point wide (27768, 23) City-month (2496, 39)


,field,dtype,definition_and_handling
0,city,str,Standardized city name parsed from filename; s...
1,state,str,State/territory mapped from the recognized cit...
2,district,object,Unavailable in source; intentionally null.
3,date,datetime64[us],Month-start timestamp identifying the monthly ...
4,rainfall_mm,float64,PRECTOTCORR monthly mean daily precipitation r...
5,avg_temp_c,float64,"T2M monthly average 2 m temperature, °C."
6,max_temp_c,float64,T2M_MAX monthly mean of daily maximum temperat...
7,min_temp_c,float64,T2M_MIN monthly mean of daily minimum temperat...
8,humidity_pct,float64,Unavailable. QV2M is specific humidity and is ...
9,aqi,float64,Unavailable in NASA/POWER files.


## Spatial aggregation choices

For comparable monthly NASA/POWER grid variables, use the arithmetic mean across available measurement points and retain coverage counts. Representative coordinates are the means of source point coordinates. Max/min temperature fields represent point-averaged monthly means of daily extrema, not the most extreme city point. AQI is not averaged because it is unavailable.

In [3]:
results=run_pipeline(RAW_DIR,ROOT)
city_monthly=results["city_monthly"]
display(city_monthly[["location_id","city","district","state","date","latitude","longitude",
 "measurement_points_available","expected_points","rainfall_mm","avg_temp_c","max_temp_c","min_temp_c",
 "rainfall_anomaly_mm","temperature_anomaly_c"]].head(12))
display(city_monthly[["city","state","location_id"]].drop_duplicates().sort_values("city"))


,location_id,city,district,state,date,latitude,longitude,measurement_points_available,expected_points,rainfall_mm,avg_temp_c,max_temp_c,min_temp_c,rainfall_anomaly_mm,temperature_anomaly_c
0,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-01-01,22.954556,72.648433,9,9,0.000000,21.238889,34.030000,6.078889,-0.027350,1.284444
1,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-02-01,22.954556,72.648433,9,9,0.000000,21.844444,38.465556,8.877778,-0.032650,-1.585000
2,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-03-01,22.954556,72.648433,9,9,0.000000,26.832222,42.344444,12.636667,-0.132521,-1.421538
3,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-04-01,22.954556,72.648433,9,9,0.000000,33.333333,45.368889,21.600000,-0.087350,0.646923
4,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-05-01,22.954556,72.648433,9,9,1.674444,34.041111,45.582222,24.572222,1.249060,-0.911581
5,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-06-01,22.954556,72.648433,9,9,1.560000,34.614444,44.198889,27.241111,-2.098291,1.222265
6,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-07-01,22.954556,72.648433,9,9,11.196667,29.012222,37.371111,22.863333,2.170556,-0.550427
7,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-08-01,22.954556,72.648433,9,9,3.330000,28.591111,35.195556,23.438889,-4.441282,0.544060
8,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-09-01,22.954556,72.648433,9,9,0.162222,30.491111,40.681111,22.680000,-4.294274,2.495812
9,IN-GJ-AMD,Ahmedabad,<NA>,Gujarat,2000-10-01,22.954556,72.648433,9,9,0.032222,30.655556,41.570000,18.217778,-0.433034,3.607094


,city,state,location_id
0,Ahmedabad,Gujarat,IN-GJ-AMD
312,Bengaluru,Karnataka,IN-KA-BLR
624,Bhopal,Madhya Pradesh,IN-MP-BHO
936,Bhubaneswar,Odisha,IN-OD-BBS
1248,Chandigarh,Chandigarh (Union Territory),IN-CH-IXC
1560,Chennai,Tamil Nadu,IN-TN-MAA
1872,Coimbatore,Tamil Nadu,IN-TN-CJB
2184,Delhi,Delhi (National Capital Territory),IN-DL-DEL


## Derived features

Location IDs are deterministic (`IN-{state code}-{city code}`). Monthly anomalies subtract the city and calendar-month mean across available years (2000–2025), with a minimum of ten valid years. The precipitation anomaly retains mm/day units. These are dataset-relative monthly anomalies, not daily climatological anomalies.